In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


## 데이터 불러오기

In [ ]:
import pandas as pd

df_edges = pd.read_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges.csv")
df_nodes = pd.read_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes.csv")

### 문제 상황
1. 호선 표기 방식 불일치

- 문제 내용:
현재 edges의 "호선" 컬럼에는 "2호선" 형태로 단일 값이 들어가 있지만,
"station" 컬럼에 "성수(1)", "신도림(2)"처럼 다중 호선 연결을 암시하는 표기가 존재합니다.
이 경우 호선 검색 시 "in" 조건으로 탐색이 불가능해집니다.

- 개선 방향:
"station" 컬럼에서 (1) 등 괄호 숫자 표기를 제거합니다.
"호선" 컬럼에는 다중 호선이 존재할 경우 콤마로 구분된 형태로 표기합니다.
예)

  station = "신도림"  
  호선 = "1호선, 2호선"


  2호선 지선의 경우 “2호선(신정지선)” 형태로 명확히 표시합니다.

  위 수정은 nodes와 edges 모두 동일하게 반영합니다.


2. 엣지(edge) 구조 불완전

- 문제 내용:
relation 값이 "환승통로"인 엣지 중,
승강장과 연결된 환승통로가 일부 방면(상/하행)에만 존재하는 경우가 있습니다.
즉, 동일 역 내에서 모든 방면에 대해 환승통로 엣지가 생성되어야 하는데
현재 일부 누락되어 양방향 연결이 불완전한 상태입니다.

- 개선 방향: 각 station 내에서 "relation" == "환승통로" 인 엣지를 검증하여,
방면별(방면_탑승구) 균형이 맞지 않는 엣지를 추가 생성합니다. 누락된 연결은 승강장-환승통로 간 양방향(source ↔ target)으로 보완합니다.


3. 특정 역 구조 데이터(신도림, 성수)

- 문제 내용:
두 역은 다중 환승 구조와 분기선이 복잡하게 얽혀 있어
현재 데이터 구조로는 올바른 연결 관계가 표현되지 않습니다.

- 개선 방향:
신도림, 성수 두 역의 nodes 및 edges 데이터를 완전히 재작성(재정의) 합니다.
각 층별 대합실·환승통로·승강장·출구를 명확히 구분하고
실제 환승 동선 기준으로 edge를 재구성합니다.

### 1.호선/역명 표기 통일

호선 컬럼(호선)

환승역: 다중 호선을 콤마로 구분해 표기합니다.
예) 2호선, 7호선

단일 호선 역: 해당 호선만 표기합니다.
예) 2호선

지선: 지선 정보를 괄호로 명시합니다.
예) 2호선(신정지선), 2호선(성수지선)

역명 컬럼(station)

괄호 등 부가 표기는 제거하고 역 이름만 남깁니다.
예) 성수(1) → 성수, 신도림(2) → 신도림


In [ ]:
import re
import pandas as pd

def normalize_line_token(tok: str)
    tok = tok.strip()
    if tok.isdigit():            # '4' -> '4호선'
        return f"{tok}호선"
    if tok.endswith("선"):       # '신림선' 그대로
        return tok
    return f"{tok}선"            # '경의중앙' -> '경의중앙선', '수인분당' -> '수인분당선'

def split_lines_field(s: str)
    # '2호선, 5호선' -> ['2호선','5호선']; NaN 대응
    if pd.isna(s) or not str(s).strip():
        return []
    return [x.strip() for x in str(s).split(",") if x.strip()]

def process_row(row):
    station_raw = str(row["station"])
    line_raw    = row["호선"]

    # 역이름(괄호내용) 패턴 추출
    m = re.match(r"^\s*([^()]+?)\s*(?:\(([^)]*)\))?\s*$", station_raw)
    if not m:
        # 매칭 안 되면 그대로
        return pd.Series([line_raw, station_raw])

    pure_name = m.group(1).strip()                 # 괄호 없는 역 이름
    inside    = m.group(2)                         # 괄호 안 내용 "4,5" / "5,경의중앙,수인분당" 등

    # 기존 호선들(있는 그대로 유지: '2호선(신정지선)' 같은 표기 보존)
    merged = []
    seen = set()
    for item in split_lines_field(line_raw):
        if item not in seen:
            merged.append(item)
            seen.add(item)

    # 괄호 안 토큰들을 정규화해서 추가
    if inside:
        for tok in inside.split(","):
            norm = normalize_line_token(tok)
            if norm not in seen:
                merged.append(norm)
                seen.add(norm)

    # 최종 문자열
    new_line = ", ".join(merged) if merged else line_raw
    new_station = pure_name
    return pd.Series([new_line, new_station])

df_edges[["호선", "station"]] = df_edges.apply(process_row, axis=1)
df_nodes[["호선", "station"]] = df_nodes.apply(process_row, axis=1)

In [ ]:
# df_nodes.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정1.csv", index=False, encoding="cp949")
# df_edges.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정1.csv", index=False, encoding="cp949")

### 2. 환승통로 엣지 연결 불균형

문제점
승강장이 방면별로 각각 노드로 구성되어 있음 (예:
2호선 ○○방면 승강장, 2호선 △△방면 승강장,
7호선 ○○방면 승강장, 7호선 △△방면 승강장).
그러나 환승통로가 한쪽 방면 승강장이랑만 엣지로 연결되어 있고,
반대 방면 승강장과는 연결이 누락되어 있음.

예시
환승통로가 대합실과 연결된 경우, 실제로는
양쪽 방면 승강장 모두와 연결되어야 하지만
현재는 한쪽 방면만 연결되어 있음.

필요한 조치
환승통로가 특정 승강장과 연결되어 있다면
같은 역 내 반대 방면 승강장과도 동일한 엣지를 추가해야 함.
즉, 환승통로 ↔ 승강장 관계는 양방향 + 양방면 모두 존재해야 함.

In [ ]:
df_nodes = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정1.csv', encoding="cp949")
df_edges = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정1.csv', encoding="cp949")

In [ ]:
to_drop_cond = (
    ((df_edges["station"] == "사당") & (df_edges["relation"] == "환승통로") &
     (df_edges["source"] == "사당_005") & (df_edges["target"] == "사당_006"))
    |
    ((df_edges["station"] == "사당") & (df_edges["relation"] == "환승통로") &
     (df_edges["source"] == "사당_006") & (df_edges["target"] == "사당_005"))
)
df_edges = df_edges.loc[~to_drop_cond].reset_index(drop=True)


new_rows = [
    {"호선": '2호선, 3호선', "station": '을지로3가', "source": '을지로2_003', "target": '을지로2_005', "relation": '환승통로'},
    {"호선": '2호선, 3호선', "station": '을지로3가', "source": '을지로2_005', "target": '을지로2_003', "relation": '환승통로'},

    {"호선": '2호선, 5호선', "station": '을지로4가', "source": '을지로3_003', "target": '을지로3_006', "relation": '환승통로'},
    {"호선": '2호선, 5호선', "station": '을지로4가', "source": '을지로3_006', "target": '을지로3_003', "relation": '환승통로'},

    {"호선": '2호선, 4호선, 5호선', "station": '동대문역사문화공원', "source": '동대문_010', "target": '동대문_011', "relation": '환승통로'},
    {"호선": '2호선, 4호선, 5호선', "station": '동대문역사문화공원', "source": '동대문_011', "target": '동대문_010', "relation": '환승통로'},
    {"호선": '2호선, 4호선, 5호선', "station": '동대문역사문화공원', "source": '동대문_009', "target": '동대문_012', "relation": '환승통로'},
    {"호선": '2호선, 4호선, 5호선', "station": '동대문역사문화공원', "source": '동대문_012', "target": '동대문_009', "relation": '환승통로'},
    {"호선": '2호선, 4호선, 5호선', "station": '동대문역사문화공원', "source": '동대문_010', "target": '동대문_012', "relation": '환승통로'},
    {"호선": '2호선, 4호선, 5호선', "station": '동대문역사문화공원', "source": '동대문_012', "target": '동대문_010', "relation": '환승통로'},

    {"호선": '2호선, 5호선, 경의중앙선, 수인분당선', "station": '왕십리', "source": '왕십리_003', "target": '왕십리_008', "relation": '환승통로'},
    {"호선": '2호선, 5호선, 경의중앙선, 수인분당선', "station": '왕십리', "source": '왕십리_008', "target": '왕십리_003', "relation": '환승통로'},
    {"호선": '2호선, 5호선, 경의중앙선, 수인분당선', "station": '왕십리', "source": '왕십리_005', "target": '왕십리_014', "relation": '환승통로'},
    {"호선": '2호선, 5호선, 경의중앙선, 수인분당선', "station": '왕십리', "source": '왕십리_014', "target": '왕십리_005', "relation": '환승통로'},
    {"호선": '2호선, 5호선, 경의중앙선, 수인분당선', "station": '왕십리', "source": '왕십리_005', "target": '왕십리_015', "relation": '환승통로'},
    {"호선": '2호선, 5호선, 경의중앙선, 수인분당선', "station": '왕십리', "source": '왕십리_015', "target": '왕십리_005', "relation": '환승통로'},

    {"호선": '2호선, 7호선', "station": '건대입구', "source": '건대입_001', "target": '건대입_006', "relation": '환승통로'},
    {"호선": '2호선, 7호선', "station": '건대입구', "source": '건대입_006', "target": '건대입_001', "relation": '환승통로'},

    {"호선": '2호선, 8호선', "station": '잠실', "source": '잠실_003', "target": '잠실_006', "relation": '환승통로'},
    {"호선": '2호선, 8호선', "station": '잠실', "source": '잠실_006', "target": '잠실_003', "relation": '환승통로'},
    {"호선": '2호선, 8호선', "station": '잠실', "source": '잠실_005', "target": '잠실_004', "relation": '환승통로'},
    {"호선": '2호선, 8호선', "station": '잠실', "source": '잠실_004', "target": '	잠실_005', "relation": '환승통로'},
    {"호선": '2호선, 8호선', "station": '잠실', "source": '잠실_006', "target": '잠실_004', "relation": '환승통로'},
    {"호선": '2호선, 8호선', "station": '잠실', "source": '잠실_004', "target": '잠실_006', "relation": '환승통로'},

    {"호선": '2호선, 분당선', "station": '선릉', "source": '선릉_004', "target": '선릉_003', "relation": '환승통로'},
    {"호선": '2호선, 분당선', "station": '선릉', "source": '선릉_003', "target": '선릉_004', "relation": '환승통로'},

    {"호선": '2호선, 신분당선', "station": '강남', "source": '강남_003', "target": '강남_004', "relation": '환승통로'},
    {"호선": '2호선, 신분당선', "station": '강남', "source": '강남_004', "target": '강남_003', "relation": '환승통로'},

    {"호선": '2호선, 3호선', "station": '교대', "source": '교대_004', "target": '교대_007', "relation": '환승통로'},
    {"호선": '2호선, 3호선', "station": '교대', "source": '교대_007', "target": '교대_004', "relation": '환승통로'},

    {"호선": '2호선, 4호선', "station": '사당', "source": '사당_005', "target": '사당_003', "relation": '환승통로'},
    {"호선": '2호선, 4호선', "station": '사당', "source": '사당_003', "target": '사당_005', "relation": '환승통로'},
    {"호선": '2호선, 4호선', "station": '사당', "source": '사당_004', "target": '사당_006', "relation": '환승통로'},
    {"호선": '2호선, 4호선', "station": '사당', "source": '사당_006', "target": '사당_004', "relation": '환승통로'},
    {"호선": '2호선, 4호선', "station": '사당', "source": '사당_004', "target": '사당_007', "relation": '환승통로'},
    {"호선": '2호선, 4호선', "station": '사당', "source": '사당_007', "target": '사당_004', "relation": '환승통로'},

    {"호선": '2호선, 5호선', "station": '영등포구청', "source": '영등포_005', "target": '영등포_006', "relation": '환승통로'},
    {"호선": '2호선, 5호선', "station": '영등포구청', "source": '영등포_006', "target": '영등포_005', "relation": '환승통로'},

    {"호선": '2호선, 6호선', "station": '합정', "source": '합정_003', "target": '합정_005', "relation": '환승통로'},
    {"호선": '2호선, 6호선', "station": '합정', "source": '합정_005', "target": '합정_003', "relation": '환승통로'},

    {"호선": '2호선, 경의중앙선, 공항철도선', "station": '홍대입구', "source": '홍대입_002', "target": '홍대입_001', "relation": '환승통로'},
    {"호선": '2호선, 경의중앙선, 공항철도선', "station": '홍대입구', "source": '홍대입_001', "target": '홍대입_002', "relation": '환승통로'},
 ]

df_new = pd.DataFrame(new_rows)

df_edges = pd.concat([df_edges, df_new], ignore_index=True)

df_edges = df_edges.drop_duplicates(
    subset=["호선", "station", "source", "target", "relation"]
).reset_index(drop=True)

## 신도림, 성수 두 역의 nodes 및 edges 데이터를 완전히 재작성
- 신도림

In [ ]:
import re
import pandas as pd

# 1) 신도림의 기존 2개 노드 삭제
cond_del = (
    (df_nodes["station"] == "신도림") &
    (df_nodes["node_id"].isin(["신도림_007", "신도림_008"])) &
    (df_nodes["type"] == "승강장")
)
df_nodes = df_nodes.loc[~cond_del].reset_index(drop=True)

# 2) 추가할 4개 노드 구성
#    - 호선은 삭제한 행의 값이 있으면 그걸 그대로 사용(보통 "2호선, 1호선")
#    - 없으면 기본값으로 "2호선, 1호선"
#    - floor/type/station 은 기존 행에서 가져오되, 기본도 지정

base_line = "2호선, 1호선"

base_floor = "지상1층"
base_type  = "승강장"
base_station = "신도림"

new_names = [
    "1호선 용산 방면 승강장",
    "1호선 서울역·소요산 방면 승강장",
    "1호선 동인천(급행)·광명 방면 승강장",
    "1호선 인천·신창(급행) 방면 승강장",
]


# 3) 신도림 node_id 자동 배정 (기존 최대 번호 다음부터)
pat = re.compile(r"^신도림_(\d+)$")
existing_nums = (
    df_nodes.loc[df_nodes["node_id"].str.match(pat.pattern, na=False), "node_id"]
      .str.extract(pat)[0]
      .dropna()
      .astype(int)
)
start_num = (existing_nums.max() if not existing_nums.empty else 0) + 1

def make_id(k):
    # 3자리 패딩 (원본 형식을 유지하고 싶으면 자릿수 맞춰 사용)
    return f"신도림_{start_num + k:03d}"

rows_to_add = []
for i, nm in enumerate(new_names):
    rows_to_add.append({
        "호선": base_line,
        "station": base_station,
        "node_id": make_id(i),
        "node_name": nm,
        "floor": base_floor,
        "type": base_type,
    })

df_nodes = pd.concat([df_nodes, pd.DataFrame(rows_to_add)], ignore_index=True)


# 4) 신도림 구간 중복 제거
#    - '호선'만 제외하고 나머지 컬럼이 모두 같으면 1개만 유지
cols_except_line = ['station', 'node_id', 'node_name', 'floor', 'type']

mask_sdr = df_nodes['station'] == '신도림'
sdr = df_nodes.loc[mask_sdr].drop_duplicates(subset=cols_except_line, keep='first')
df_nodes = pd.concat([df_nodes.loc[~mask_sdr], sdr], ignore_index=True)

In [ ]:
import pandas as pd

# 1) 신도림 관련 삭제
to_delete_ids = [
    "신도림_006",
    "신도림2_001", "신도림2_002", "신도림2_003", "신도림2_004",
    "신도림2_007", "신도림2_008", "신도림2_009", "신도림2_010",
]

mask_delete = (df_nodes["station"] == "신도림") & (df_nodes["node_id"].isin(to_delete_ids))
df_nodes = df_nodes.loc[~mask_delete].reset_index(drop=True)

# 2) 신도림2_005 / 신도림2_006 -> id/이름 변경
#    - id 재사용: 001, 002
updates = {
    "신도림2_005": {"node_id": "신도림_006", "node_name": "2호선(신정지선) 대림 방면 승강장"},
    "신도림2_006": {"node_id": "신도림_007", "node_name": "2호선(신정지선) 도림천 방면 승강장"},
}

for old_id, new_vals in updates.items():
    idx = df_nodes.index[(df_nodes["station"] == "신도림") & (df_nodes["node_id"] == old_id)]
    if len(idx) == 1:
        i = idx[0]
        df_nodes.at[i, "node_id"] = new_vals["node_id"]
        df_nodes.at[i, "node_name"] = new_vals["node_name"]
        # 나머지 컬럼(호선/층/타입)은 기존 값 유지
    elif len(idx) == 0:
        print(f"[경고] 대상 행을 찾지 못함: {old_id}")
    else:
        print(f"[경고] 대상 행이 여러 개 발견됨: {old_id} -> {list(idx)}")

# 3) node_id 중복이 없도록 안전 확인(필요 시 실패)
dup_ids = df_nodes[df_nodes["node_id"].duplicated()]["node_id"].unique()
if len(dup_ids) > 0:
    raise ValueError(f"node_id 중복 발견: {dup_ids}")

df_nodes.loc[df_nodes['station'] == '신도림', '호선'] = '2호선, 2호선(신정지선), 1호선'

In [ ]:
# 1) 신도림 행 모두 삭제
df_edges = df_edges[df_edges['station'] != '신도림'].reset_index(drop=True)

# 2) 새 행 생성 (질문에 준 new_lst 사용)
new_lst = [
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_009', 'relation': '출구->대합실'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_009', 'target': '신도림_001', 'relation': '출구->대합실'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_004', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_004', 'target': '신도림_001', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_011', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_011', 'target': '신도림_001', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_012', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_012', 'target': '신도림_001', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_013', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_013', 'target': '신도림_001', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_014', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_014', 'target': '신도림_001', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_002', 'target': '신도림_010', 'relation': '출구->대합실'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_010', 'target': '신도림_002', 'relation': '출구->대합실'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_002', 'target': '신도림_003', 'relation': '대합실->대합실'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_003', 'target': '신도림_002', 'relation': '대합실->대합실'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_002', 'target': '신도림_011', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_011', 'target': '신도림_002', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_002', 'target': '신도림_012', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_012', 'target': '신도림_002', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_002', 'target': '신도림_013', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_013', 'target': '신도림_002', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_002', 'target': '신도림_014', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_014', 'target': '신도림_002', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_004', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_004', 'target': '신도림_001', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_005', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_005', 'target': '신도림_001', 'relation': '대합실->승강장'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_006', 'target': '신도림_001', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_006', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_007', 'target': '신도림_001', 'relation': '환승통로'},
    {'호선': '2호선, 2호선(신정지선), 1호선','station': '신도림', 'source': '신도림_001', 'target': '신도림_007', 'relation': '환승통로'},
]

new_df = pd.DataFrame(new_lst)

# 3) 공백 정리(혹시 모를 탭/스페이스) 및 컬럼 맞추기
for col in ['호선','station','source','target','relation']:
    new_df[col] = new_df[col].astype(str).str.strip()

# df_edges의 컬럼 순서에 맞춤 (추가/누락 안전 처리)
for c in df_edges.columns:
    if c not in new_df.columns:
        new_df[c] = np.nan
new_df = new_df[df_edges.columns]

# 4) 합치기 + (선택) 중복 제거 + 정렬
df_edges = pd.concat([df_edges, new_df], ignore_index=True)

# 중복 제거(완전 동일 엣지 기준)
df_edges = df_edges.drop_duplicates(subset=['호선','station','source','target','relation']).reset_index(drop=True)

# 보기 좋게 정렬(옵션)
df_edges = df_edges.sort_values(['station','source','target']).reset_index(drop=True)

## node_name에 (지하 ~~)등 층 정보가 있으면 전처리(floor에 층 정보가 표기되어있으므로)

In [ ]:
df_nodes['node_name'] = (
    df_nodes['node_name']
      .astype(str)
      .str.replace(r"\(지하[^)]*\)", "", regex=True)  # '(지하…)' 만 제거
      .str.replace(r"\s{2,}", " ", regex=True)        # 중복 공백 정리
      .str.strip()
)

- 성수

In [ ]:
# 1) 성수 행에서 source/target의 '-' 제거
mask_ss = df_edges['station'].eq('성수')
for col in ['source', 'target']:
    df_edges.loc[mask_ss, col] = (
        df_edges.loc[mask_ss, col]
            .astype(str)
            .str.replace('-', '', regex=False)
            .str.strip()
    )

# 2) 호선이 정확히 '2호선(성수지선)'인 행 제거
df_edges = df_edges.loc[~df_edges['호선'].eq('2호선(성수지선)')].reset_index(drop=True)

# 3) 성수 행들의 '호선' 값을 '2호선, 2호선(성수지선)'으로 일괄 수정
mask_ss = df_edges['station'].eq('성수')  # (삭제 후 다시 잡기)
df_edges.loc[mask_ss, '호선'] = '2호선, 2호선(성수지선)'

# 4) 지정된 2개 엣지의 relation을 '환승통로'로 변경
#    (앞에서 하이픈을 제거했으므로 '성수_001' <-> '성수_004' 기준으로 매칭)
pairs_to_update = [
    ('성수_001', '성수_004'),
    ('성수_004', '성수_001'),
]
for s, t in pairs_to_update:
    cond = (df_edges['station'].eq('성수') &
            df_edges['source'].eq(s) &
            df_edges['target'].eq(t))
    df_edges.loc[cond, 'relation'] = '환승통로'


# 1) 성수 행에서 node_id의 '-' 제거
mask_ss_n = df_nodes['station'].eq('성수')
df_nodes.loc[mask_ss_n, 'node_id'] = (
    df_nodes.loc[mask_ss_n, 'node_id']
        .astype(str)
        .str.replace('-', '', regex=False)
        .str.strip()
)

# 2) 아래 5개(성수-2_001~005) 행 삭제
#    하이픈 제거 전/후 모두 커버하도록 정규식 사용: ^성수-?2_00[1-5]$
del_mask = (
    mask_ss_n &
    df_nodes['node_id'].astype(str).str.contains(r'^성수-?2_00[1-5]$', regex=True)
)
df_nodes = df_nodes.loc[~del_mask].reset_index(drop=True)

# 3) 성수 행들의 '호선' 값을 '2호선, 2호선(성수지선)'으로 일괄 수정
mask_ss_n = df_nodes['station'].eq('성수')  # (삭제 후 다시 잡기)
df_nodes.loc[mask_ss_n, '호선'] = '2호선, 2호선(성수지선)'

# 4) node_id='성수_004'의 node_name 문구 변경
cond_004 = (
    df_nodes['station'].eq('성수') &
    df_nodes['node_id'].eq('성수_004') &
    df_nodes['node_name'].astype(str).str.contains('2호선 용답 방면 승강장')
)
df_nodes.loc[cond_004, 'node_name'] = '2호선(성수지선) 용답 방면 승강장'

In [ ]:
# df_nodes.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정2.csv", index=False, encoding="cp949")
# df_edges.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정2.csv", index=False, encoding="cp949")

## 업그레이드
1) Nodes

출구 노드 분리

현재 출구 노드가 묶음으로 되어 있음 → 각 출구를 개별 노드로 생성해야 함

이에 따라 엣지도 각 출구에 맞춰 개별 생성 필요

방면별 추천 탑승구 노드 생성

각 방면마다 빠른하차/빠른환승 탑승구 노드 생성

생성에 맞춰 방면 승강장 ↔ 탑승구 양방향 엣지 추가
(예: xx방면 승강장 → x-x 탑승구, x-x 탑승구 → xx방면 승강장)

2) Edges

에스컬레이터 유무 컬럼 추가

각 엣지에 에스컬레이터 유무 정보를 심어 컬럼으로 관리

relation 상세 문구화

안내멘트를 relation 한 줄로 제공할 예정이므로,
기존의 단순 표기(예: 대합실→승강장)에서 상세 문구로 확장

예: 지하 x층 대합실에서 에스컬레이터를 타고 xx방면 승강장으로 이동하세요.

환승통로도 단순 환승통로가 아니라 출발/도착 지점과 이동수단을 포함해 표기
(예: xx승강장에서 x호선 대합실로 환승통로를 통해 에스컬레이터로 이동하세요.)

3) ERD 관련

코드 체계 부여

노드 코드 / 엣지 코드 / 역 코드 생성

추후 PK/FK 설정을 위해 코드 체계 확립 필요

## 출구 분리

In [ ]:
# df_nodes = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정2.csv', encoding="cp949")
# df_edges = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정2.csv', encoding="cp949")

In [ ]:
# 업데이트: '출구(1~11)' 또는 혼합형 '출구(1~3,5,7~9)'까지 처리
import re
import pandas as pd
from pathlib import Path

def read_csv_with_fallback(path: str):
    for enc in ["utf-8", "utf-8-sig", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)

nodes_path = Path("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정2.csv")
edges_path = Path("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정2.csv")

out_nodes_path = Path("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정3.csv")
out_edges_path = Path("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정3.csv")
log_path = Path("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/split_log.csv")

df_nodes = read_csv_with_fallback(nodes_path)
df_edges = read_csv_with_fallback(edges_path)

# 핵심: 출구 리스트 파서 (콤마/범위 혼합 지원)
# 허용 구분자: 콤마(,)，중국어 콤마(，)와 범위 구분자(~, -, –, —)
range_sep = r"[~\-–—]"
comma_sep = r"[,\uFF0C]"  # ',' 와 '，'

def parse_exit_numbers(text: str):
    """
    '출구(1,2,11,12)' -> [1,2,11,12]
    '출구(1~11)' -> [1,2,...,11]
    '출구(1~3,5,7~9)' -> [1,2,3,5,7,8,9]
    공백 자유, 중복 제거+원래 순서 유지
    """
    # 괄호 안만 추출
    m = re.search(r"\(\s*([^)]+)\s*\)", str(text))
    if not m:
        return []
    inner = m.group(1)

    parts = re.split(comma_sep, inner)  # 콤마 분리
    seq = []
    seen = set()

    for part in parts:
        part = part.strip()
        if not part:
            continue
        # 범위?
        if re.search(range_sep, part):
            # 'a~b' 형태에서 양쪽 숫자 추출
            nums = re.split(range_sep, part)
            if len(nums) == 2 and nums[0].strip().isdigit() and nums[1].strip().isdigit():
                a, b = int(nums[0].strip()), int(nums[1].strip())
                if a <= b:
                    rng = list(range(a, b + 1))
                else:
                    # 잘못된 역방향 범위가 오면 swap
                    rng = list(range(b, a + 1))
                for v in rng:
                    if v not in seen:
                        seq.append(v); seen.add(v)
            # 형식이 틀리면 무시
        else:
            # 단일 숫자
            if part.isdigit():
                v = int(part)
                if v not in seen:
                    seq.append(v); seen.add(v)
    return seq

# 패턴: '출구(...)' 만 처리
pat_exit_group = re.compile(r"^\s*출구\s*\(\s*.+\s*\)\s*$")

rows_to_expand = df_nodes[df_nodes["node_name"].astype(str).str.match(pat_exit_group)].copy()

# node_id 파싱
id_pattern = re.compile(r"^(.*_)(\d+)$")
def split_id(node_id: str):
    m = id_pattern.match(str(node_id))
    return (m.group(1), int(m.group(2))) if m else (None, None)

# 전역 중복 방지용
all_existing_ids = set(map(str, df_nodes["node_id"].astype(str)))

def next_free_ids(prefix: str, start_from: int, k: int):
    found = []
    cand = start_from + 1
    while len(found) < k:
        cand_id = f"{prefix}{cand:03d}"
        if cand_id not in all_existing_ids:
            found.append(cand)
            all_existing_ids.add(cand_id)
        cand += 1
    return found

new_nodes = []
mapping = {}
logs = []

for _, row in rows_to_expand.iterrows():
    orig_id = str(row["node_id"])
    name = str(row["node_name"])

    exits = parse_exit_numbers(name)
    if not exits:
        continue

    prefix, tail = split_id(orig_id)
    if prefix is None:
        continue

    same_prefix = df_nodes["node_id"].astype(str).str.startswith(prefix)
    if same_prefix.any():
        tails = df_nodes.loc[same_prefix, "node_id"].astype(str)\
                .apply(lambda x: split_id(x)[1]).dropna().astype(int)
        current_max = int(tails.max()) if len(tails) else tail
    else:
        current_max = tail

    need_new = max(0, len(exits) - 1)
    new_nums = next_free_ids(prefix, start_from=current_max, k=need_new)

    expanded_ids = [orig_id] + [f"{prefix}{n:03d}" for n in new_nums]
    mapping[orig_id] = expanded_ids

    for i, exit_no in enumerate(exits):
        new_row = row.copy()
        new_row["node_id"] = expanded_ids[i]
        new_row["node_name"] = f"{exit_no}번출구"
        new_nodes.append(new_row)

    logs.append({
        "station": row.get("station", ""),
        "original_node_id": orig_id,
        "original_node_name": name,
        "expanded_ids": "|".join(expanded_ids),
        "exits": "|".join(map(str, exits)),
    })

if not mapping:
    # 변화 없음 -> 원본 그대로 저장
    df_nodes.to_csv(out_nodes_path, index=False, encoding="utf-8-sig")
    df_edges.to_csv(out_edges_path, index=False, encoding="utf-8-sig")
    pd.DataFrame(logs).to_csv(log_path, index=False, encoding="utf-8-sig")
else:
    df_nodes_after = df_nodes.drop(index=rows_to_expand.index).copy()
    df_nodes_after = pd.concat([df_nodes_after, pd.DataFrame(new_nodes)], ignore_index=True)

    # 엣지 확장
    df_edges_after = df_edges.copy()
    for orig_id, expanded_ids in mapping.items():
        # source
        src_mask = df_edges_after["source"].astype(str) == orig_id
        if src_mask.any():
            base = df_edges_after.loc[src_mask].copy()
            expanded_src = []
            for nid in expanded_ids:
                tmp = base.copy()
                tmp["source"] = nid
                expanded_src.append(tmp)
            expanded_src_df = pd.concat(expanded_src, ignore_index=True)
            df_edges_after = pd.concat([df_edges_after.loc[~src_mask], expanded_src_df], ignore_index=True)

        # target
        tgt_mask = df_edges_after["target"].astype(str) == orig_id
        if tgt_mask.any():
            base = df_edges_after.loc[tgt_mask].copy()
            expanded_tgt = []
            for nid in expanded_ids:
                tmp = base.copy()
                tmp["target"] = nid
                expanded_tgt.append(tmp)
            expanded_tgt_df = pd.concat(expanded_tgt, ignore_index=True)
            df_edges_after = pd.concat([df_edges_after.loc[~tgt_mask], expanded_tgt_df], ignore_index=True)

    # 정렬(가독성)
    sort_cols = [c for c in ["station", "node_id"] if c in df_nodes_after.columns]
    if sort_cols:
        df_nodes_after.sort_values(by=sort_cols, kind="stable", inplace=True, ignore_index=True)

    # 저장
    df_nodes_after.to_csv(out_nodes_path, index=False, encoding="utf-8-sig")
    df_edges_after.to_csv(out_edges_path, index=False, encoding="utf-8-sig")
    pd.DataFrame(logs).to_csv(log_path, index=False, encoding="utf-8-sig")

summary = {
    "분리 대상 노드 수": len(rows_to_expand),
    "노드 수(원본)": len(read_csv_with_fallback(nodes_path)),
    "노드 수(분리후)": len(pd.read_csv(out_nodes_path, encoding='utf-8-sig')),
    "엣지 수(원본)": len(read_csv_with_fallback(edges_path)),
    "엣지 수(분리후)": len(pd.read_csv(out_edges_path, encoding='utf-8-sig')),
}
summary


{'분리 대상 노드 수': 82,
 '노드 수(원본)': 353,
 '노드 수(분리후)': 660,
 '엣지 수(원본)': 706,
 '엣지 수(분리후)': 1330}

In [ ]:
# df_nodes3 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정3.csv')
# df_edges3 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정3.csv')
# log_csv = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/split_log.csv')

## 탑승구 노드 및 엣지 추가

In [ ]:
# df_nodes3 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정3.csv')
# df_edges3 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정3.csv')
# df_빠른하차 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/df_빠른하차.csv')
# df_층 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/탑승구_층.csv')

In [ ]:
df_빠른하차 = df_빠른하차[['호선', '역명', '상하행', '방면', '탑승구', '근접이동시설', '방면_탑승구']]
df_빠른하차 = df_빠른하차.rename(columns = {'역명':'station'})
df_층 = df_층.rename(columns = {'역명':'station'})

In [ ]:
## 역명 리스트
station_list = list(df_nodes3['station'].unique())

In [ ]:
# 원하는 역과 컬럼만 필터링
탑승구 = df_빠른하차[df_빠른하차['station'].isin(station_list)][['호선','station', '방면', '탑승구', '근접이동시설']]

# df_층 호선에 숫자만 남기기(merge위한 전처리)
df_층['호선'] = df_층['호선'].str.extract(r'(\d+)').astype('int')

# df_층과 merge 해서 층 정보 추가
merged = 탑승구.merge(
    df_층[['호선', 'station', '플랫폼층_int']],  # 필요한 컬럼만 선택
    on=['호선', 'station'],                    # 공통 키 기준
    how='left'                                 # df_탑승구 중심으로 병합
)

In [ ]:
# 엘리베이터제외(이유: 다른 동선엔 엘리베이터 고려안하는데 여기서만 고려하는건 통일성이 없다고 판단)
merged = merged[merged['근접이동시설'] != '엘리베이터']
merged = merged.drop_duplicates()

# 탑승구 묶음하나로 합치기
merged_grouped = (
    merged
    .groupby(['호선', 'station', '방면', '근접이동시설', '플랫폼층_int'], as_index=False)
    .agg({'탑승구': ', '.join})
)
merged_grouped

,호선,station,방면,근접이동시설,플랫폼층_int,탑승구
0,1,시청,서울역,계단,-2,"5-4, 7-4, 9-3"
1,1,시청,종각,계단,-2,"2-2, 4-1, 6-1, 7-4"
2,1,신설동,동묘앞,계단,-2,2-1
3,1,신설동,제기동,계단,-2,9-4
4,2,강남,교대,계단,-2,"2-3, 4-2, 7-2, 9-1"
...,...,...,...,...,...,...
168,7,대림,남구로,에스컬레이터,-2,1-1
169,7,대림,신풍,계단,-2,"4-1, 5-4, 8-4"
170,7,대림,신풍,에스컬레이터,-2,8-4
171,8,잠실,몽촌토성,계단,-3,"3-1, 5-1"


In [ ]:
import pandas as pd
import re
import numpy as np

mg = merged_grouped.copy()

# --------- 1) 역별 기존 node_id 최대값 추출 (겹치지 않도록) ---------
def max_suffix_for_station(df_nodes, stn):
    pattern = rf'^{re.escape(stn)}_(\d+)$'
    nums = (
        df_nodes.loc[df_nodes['station'] == stn, 'node_id']
        .astype(str)
        .str.extract(pattern)[0]
        .dropna()
    )
    return nums.astype(int).max() if not nums.empty else 0

station_max = {stn: max_suffix_for_station(df_nodes3, stn)
               for stn in df_nodes3['station'].unique()}

# --------- 2) 역별 df_nodes3의 모든 호선 따라가기 (여러 호선이면 모두 추가) ---------
line_map_all = (
    df_nodes3.groupby('station')['호선']
    .apply(lambda s: list(pd.unique(s.dropna())))
    .to_dict()
)

# --------- 3) 플랫폼층_int → floor 문자로 변환 ---------
def floor_label(x):
    if pd.isna(x):
        return pd.NA
    try:
        v = int(x)
    except Exception:
        return pd.NA
    if v < 0:
        return f"지하{abs(v)}층"
    elif v > 0:
        return f"지상{v}층"

mg['floor'] = mg['플랫폼층_int'].apply(floor_label)

# --------- 4) 새 탑승구 노드 생성 ---------
rows = []
counters = {}  # 역별로 이번에 새로 만든 개수 누적

for _, r in mg.iterrows():
    stn = r['station']
    gates = r['탑승구']
    direction = r['방면']
    node_name_line = r.get('호선', "")
    floor_str = r.get('floor', pd.NA)

    # df_nodes3에 기록된 해당 역의 "모든 호선"에 대해 노드 추가
    lines_for_station = line_map_all.get(stn, [node_name_line])

    for line_df_nodes in lines_for_station:
        base = station_max.get(stn, 0)
        used = counters.get(stn, 0)
        next_num = base + used + 1
        counters[stn] = used + 1

        node_id = f"{stn}_{next_num:03d}"

        # '호선' 컬럼은 df_nodes3의 값을 그대로 따름
        # node_name은 merged_grouped의 호선/방면/탑승구를 사용
        node_name = f"{node_name_line}호선 {direction} 방면 {gates}".strip()

        row = {
            '호선': line_df_nodes,     # df_nodes3의 호선을 따름
            'station': stn,
            'node_id': node_id,        # 역별 최대값 이후 연속 번호
            'node_name': node_name,    # "{merged_grouped.호선}호선 {방면} 방면 {탑승구}"
            'type': '탑승구',
            'floor': floor_str,        # "지하2층"/"지상3층" 등
        }
        rows.append(row)

new_nodes = pd.DataFrame(rows)

# --------- 5) 스키마 맞추기 & 병합 ---------
for col in df_nodes3.columns:
    if col not in new_nodes.columns:
        new_nodes[col] = pd.NA
new_nodes = new_nodes[df_nodes3.columns]  # 컬럼 순서 맞추기

df_nodes4 = pd.concat([df_nodes3, new_nodes], ignore_index=True)

In [ ]:
import pandas as pd
import numpy as np
import re

# 0) 준비: 기본 정리 (원본 보존)
df_nodes4['station'] = df_nodes4['station'].astype(str).str.strip()
merged_grouped['station'] = merged_grouped['station'].astype(str).str.strip()

# 1) 노드 전처리: 승강장/탑승구만 추출 + 방향 키 파싱
#    node_name 예: "2호선 서울역 방면 2-3, 4-2"
#    dir_key  : "2호선 서울역 방면"
#    dir_name : "서울역"
nodes = df_nodes4[df_nodes4['type'].isin(['승강장', '탑승구'])].copy()
nodes['dir_key']  = nodes['node_name'].str.extract(r'^(\d+호선\s+.*?방면)', expand=False)
nodes['dir_name'] = nodes['node_name'].str.extract(r'^\d+호선\s+(.*?)\s*방면', expand=False)

plat = nodes[nodes['type'] == '승강장'].copy()
gate = nodes[nodes['type'] == '탑승구'].copy()

pair_keys = ['호선', 'station', 'floor', 'dir_key']
pairs = plat.merge(gate, on=pair_keys, suffixes=('_plat', '_gate'), how='inner')

# merge 접미사 보정
rename_map = {}
if 'node_id_x' in pairs.columns and 'node_id_plat' not in pairs.columns:
    rename_map['node_id_x'] = 'node_id_plat'
if 'node_id_y' in pairs.columns and 'node_id_gate' not in pairs.columns:
    rename_map['node_id_y'] = 'node_id_gate'
if 'dir_name_x' in pairs.columns and 'dir_name_plat' not in pairs.columns:
    rename_map['dir_name_x'] = 'dir_name_plat'
if 'dir_name_y' in pairs.columns and 'dir_name_gate' not in pairs.columns:
    rename_map['dir_name_y'] = 'dir_name_gate'
if 'node_name_y' in pairs.columns and 'node_name_gate' not in pairs.columns:
    rename_map['node_name_y'] = 'node_name_gate'
if rename_map:
    pairs = pairs.rename(columns=rename_map)

required_cols = ['호선','station','node_id_plat','node_id_gate','dir_name_plat','node_name_gate']
missing = [c for c in required_cols if c not in pairs.columns]
if missing:
    raise KeyError(f"[pairs]에 필요한 컬럼이 없습니다: {missing}")

# 2) merged_grouped → (station, 방면, 탑승구토큰) 에스컬레이터 룩업
def split_gates(gates_str):
    if pd.isna(gates_str):
        return []
    return [t.strip() for t in str(gates_str).split(',') if t.strip()]

mg_tmp = merged_grouped.copy()
mg_tmp['has_escal'] = mg_tmp['근접이동시설'].astype(str).str.contains('에스컬레이터', na=False)

rows = []
for _, r in mg_tmp.iterrows():
    stn = r['station']
    direction = r['방면']
    for g in split_gates(r['탑승구']):
        rows.append((stn, direction, g, r['has_escal']))

mg_escal_df = pd.DataFrame(rows, columns=['station','방면','gate','has_escal']).groupby(
    ['station','방면','gate'], as_index=False
)['has_escal'].any()

esc_lookup = {
    (stn, d, g): flag
    for stn, d, g, flag in mg_escal_df[['station','방면','gate','has_escal']].itertuples(index=False)
}

# 3) 탑승구 노드 node_name에서 개별 탑승구 토큰 추출
def extract_gates_from_node_name(name: str):
    if pd.isna(name):
        return []
    m = re.search(r'방면\s*(.*)$', str(name))
    if not m:
        return []
    return split_gates(m.group(1))


# 4) 엣지별 에스컬레이터 플래그 (탑승구까지 일치해야 1)
def edge_escalator_flag(station: str, dir_name: str, gate_node_name: str) -> int:
    for g in extract_gates_from_node_name(gate_node_name):
        if esc_lookup.get((station, dir_name, g), False):
            return 1
    return 0

# 5) 엣지 생성 (양방향)
def make_edges(df_pairs: pd.DataFrame) -> pd.DataFrame:
    e1 = pd.DataFrame({
        '호선':    df_pairs['호선'],
        'station': df_pairs['station'],
        'source':  df_pairs['node_id_plat'],
        'target':  df_pairs['node_id_gate'],
        'relation': '승강장->탑승구',
        '에스컬레이터': [
            edge_escalator_flag(st, dn, ng)
            for st, dn, ng in zip(df_pairs['station'], df_pairs['dir_name_plat'], df_pairs['node_name_gate'])
        ],
    })
    e2 = pd.DataFrame({
        '호선':    df_pairs['호선'],
        'station': df_pairs['station'],
        'source':  df_pairs['node_id_gate'],
        'target':  df_pairs['node_id_plat'],
        'relation': '승강장->탑승구',
        '에스컬레이터': [
            edge_escalator_flag(st, dn, ng)
            for st, dn, ng in zip(df_pairs['station'], df_pairs['dir_name_plat'], df_pairs['node_name_gate'])
        ],
    })
    return pd.concat([e1, e2], ignore_index=True)

new_edges = make_edges(pairs)


# 6) df_edges4 = df_edges3 복사본 + new_edges 추가 (df_edges3 불변)
df_edges4 = df_edges3.copy()

# (핵심) df_edges4에 '에스컬레이터' 컬럼이 없으면 생성 (기존 행은 None)
if '에스컬레이터' not in df_edges4.columns:
    df_edges4['에스컬레이터'] = None

# new_edges에도 '에스컬레이터'가 반드시 존재하도록 보장
if '에스컬레이터' not in new_edges.columns:
    new_edges['에스컬레이터'] = None

# 스키마 맞추기: df_edges4에 있는 컬럼이 new_edges에도 모두 있도록 채움
for col in df_edges4.columns:
    if col not in new_edges.columns:
        new_edges[col] = pd.NA

# (반대로) new_edges에만 있는 추가 컬럼이 있으면 df_edges4에도 만들어서 보존하고 싶다면:
for col in new_edges.columns:
    if col not in df_edges4.columns:
        df_edges4[col] = pd.NA

# 컬럼 순서 맞추기(동일 순서)
new_edges = new_edges[df_edges4.columns]

# 합치기
df_edges4 = pd.concat([df_edges4, new_edges], ignore_index=True)

# 중복 제거
df_edges4 = df_edges4.drop_duplicates(
    subset=['호선','station','source','target','relation'],
    keep='first'
).reset_index(drop=True)

df_edges4 = df_edges4.rename(columns = {'에스컬레이터':'다음역하차_에스컬레이터_유무'})

In [ ]:
# df_nodes4.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정4.csv", index=False)
# df_edges4.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정4.csv", index=False)

## 엣지에 에스컬레이터 정보 추가

In [ ]:
# df_nodes4 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_수정4.csv')
# df_edges4 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_수정4.csv')
# df_에스컬레이터 = pd.read_csv('/content/drive/MyDrive/project_wisheasy/안내로직_ver2/df_에스컬레이터.csv')

## 에스컬레이터와 운영이상여부는 임의 데이터 사용하겠습니다

In [ ]:
import numpy as np

np.random.seed(42)

# 0:70%, 1:30% 확률로 생성
df_edges4['에스컬레이터'] = np.random.choice([0, 1], size=len(df_edges4), p=[0.7, 0.3]).astype(int)

# 운영여부 컬럼은 전부 0으로
df_edges4['운영이상여부'] = 0

## edges에 relation 상세하게 수정

In [ ]:
import numpy as np
import pandas as pd


# 0) 준비: 노드 정보(타입/이름) 룩업 후, 엣지에 소스/타깃 양쪽 조인
nodes_key = df_nodes4[['node_id', 'type', 'node_name']].drop_duplicates()

edges = df_edges4.copy()
edges = edges.merge(nodes_key.add_prefix('source_'),
                    left_on='source', right_on='source_node_id', how='left')
edges = edges.merge(nodes_key.add_prefix('target_'),
                    left_on='target', right_on='target_node_id', how='left')


# 1) 에스컬레이터 이용 가능 여부 판단
has_err_col = '운영이상여부' in edges.columns
has_op_col  = '운영여부'   in edges.columns

if has_err_col:
    escalator_ok = (edges['에스컬레이터'].fillna(0).astype(int) == 1) & (edges['운영이상여부'].fillna(1).astype(int) == 0)
elif has_op_col:
    escalator_ok = (edges['에스컬레이터'].fillna(0).astype(int) == 1) & (edges['운영여부'].fillna(0).astype(int) == 1)
else:
    escalator_ok = (edges['에스컬레이터'].fillna(0).astype(int) == 1)

move_txt = np.where(escalator_ok, '에스컬레이터를 이용하여', '계단/도보를 이용하여')


# 2) 조건 마스크 정의
m_board  = (edges['target_type'] == '탑승구')
m_alight = (edges['source_type'] == '탑승구')
m_else   = ~(m_board | m_alight)


# 3) relation 생성 기본 문구
relation = np.empty(len(edges), dtype=object)

# A) 승차 안내
relation[m_board] = edges.loc[m_board, 'target_node_name'].astype(str) + '번 탑승구에서 승차하세요.'

# B) 하차 안내
relation[m_alight] = '하차 위치는 ' + edges.loc[m_alight, 'target_node_name'].astype(str) + '번 입니다.'

# C) 일반 이동 안내
relation[m_else] = (
    edges.loc[m_else, 'source_node_name'].astype(str)
    + '에서 '
    + move_txt[m_else]
    + ' '
    + edges.loc[m_else, 'target_node_name'].astype(str)
    + '으로 이동하세요.'
)


# 4) 환승통로 문구 추가
#    기존 relation 값이 "환승통로"였던 행이라면 문장 끝에 추가
is_transfer = edges['relation'].fillna('') == '환승통로'
relation[is_transfer] = relation[is_transfer] + ' (이 구간은 환승통로입니다.)'


# 5) 최종 반영
df_edges4['relation'] = relation

In [ ]:
import numpy as np
import pandas as pd


# 0) 준비: 노드 정보(타입/이름) 룩업 후, 엣지에 소스/타깃 양쪽 조인
nodes_key = df_nodes4[['node_id', 'type', 'node_name']].drop_duplicates()

edges = df_edges4.copy()
edges = edges.merge(nodes_key.add_prefix('source_'),
                    left_on='source', right_on='source_node_id', how='left')
edges = edges.merge(nodes_key.add_prefix('target_'),
                    left_on='target', right_on='target_node_id', how='left')


# 1) 에스컬레이터 가용 여부 플래그 만들기
#    - 기본: 에스컬레이터==1 이고 운영이상여부==0이면 "에스컬레이터를 이용하여"
#    - 컬럼명이 다를 수 있어 방어적으로 처리:
#        * '운영이상여부'가 있으면 0(이상 없음)이 가용
#        * 없고 '운영여부'가 있으면 1(운영 중)이 가용
#        * 둘 다 없으면 에스컬레이터==1만 고려
has_err_col = '운영이상여부' in edges.columns
has_op_col  = '운영여부'   in edges.columns

if has_err_col:
    escalator_ok = (edges['에스컬레이터'].fillna(0).astype(int) == 1) & (edges['운영이상여부'].fillna(1).astype(int) == 0)
elif has_op_col:
    escalator_ok = (edges['에스컬레이터'].fillna(0).astype(int) == 1) & (edges['운영여부'].fillna(0).astype(int) == 1)
else:
    escalator_ok = (edges['에스컬레이터'].fillna(0).astype(int) == 1)

# 안내 문구에서 사용할 이동수단 텍스트
move_txt = np.where(escalator_ok, '에스컬레이터를 이용하여', '계단/도보를 이용하여')


# 2) 조건 마스크 정의
#    A) 타깃 노드 타입이 '탑승구' → "OO번 탑승구에서 승차하세요."
#    B) 소스 노드 타입이 '탑승구' → "하차 위치는 OO번 입니다."
#    C) 그 외 → "AAA에서 (이동수단) BBB으로 이동하세요."
#   (A가 B보다 우선하도록 순서 주의)
m_board  = (edges['target_type'] == '탑승구')
m_alight = (edges['source_type'] == '탑승구')
m_else   = ~(m_board | m_alight)


# 3) relation 컬럼 생성
relation = np.empty(len(edges), dtype=object)

# A) 승차 안내
relation[m_board] = edges.loc[m_board, 'target_node_name'].astype(str).radd('').values + '번 탑승구에서 승차하세요.'

# B) 하차 안내
relation[m_alight] = '하차 위치는 ' + edges.loc[m_alight, 'target_node_name'].astype(str).values + ' 입니다.'

# C) 일반 이동 안내
relation[m_else] = (
    edges.loc[m_else, 'source_node_name'].astype(str).values
    + '에서 '
    + move_txt[m_else]
    + ' '
    + edges.loc[m_else, 'target_node_name'].astype(str).values
    + '으로 이동하세요.'
)


# 4) df_edges4에 반영
df_edges4['relation'] = relation

## edges에 key값 생성

In [ ]:
# edge_key 생성: "source→target"
df_edges4['edge_key'] = (
    df_edges4['source'].astype(str).str.strip() + '→' +
    df_edges4['target'].astype(str).str.strip()
)

# 중복 여부 확인 (선택)
dup_edges = df_edges4[df_edges4['edge_key'].duplicated(keep=False)]
if len(dup_edges):
    print(f"⚠️ 중복된 edge_key {len(dup_edges)}개 발견!")
    display(dup_edges)
else:
    print("모든 edge_key가 고유합니다.")

모든 edge_key가 고유합니다.


In [ ]:
# df_nodes4.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_nodes_최종.csv", index=False)
# df_edges4.to_csv("/content/drive/MyDrive/project_wisheasy/안내로직_ver2/line2_edges_최종.csv", index=False)